In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)

from sklearn.model_selection import ParameterGrid

### 데이터 불러오기

In [8]:
# 파일 불러오기 
target = "Risk_Label"

# Load datasets
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

train shape: (2339, 16) (2339,)
valid shape: (1438, 16) (1438,)
test shape: (822, 16) (822,)


## ParameterGrid SVM 적합

In [9]:
# =========================
# Parameter search space
# linear kernel에서는 gamma가 의미 없으므로 따로 분리
# =========================
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__gamma": ["scale", "auto", 0.01, 0.1],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["poly"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__gamma": ["scale", "auto", 0.01, 0.1],
        "svm__class_weight": [None],
    },
]


def compute_metrics(y_true, y_pred):
    """
    Accuracy, F1, Recall, Precision, G-Mean, H1 계산.
    H1 = F1과 G-Mean의 조화평균.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    g_mean = np.sqrt(sensitivity * specificity)

    h1 = (
        2 * f1 * g_mean / (f1 + g_mean)
        if (f1 + g_mean) > 0
        else 0.0
    )

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "g_mean": g_mean,
        "h1": h1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "cm": cm,
    }


def make_threshold_candidates(scores):
    """
    SVM decision_function score 기준 threshold 후보 생성.
    score >= threshold 이면 high-risk(1)로 분류.

    validation score의 모든 unique 값을 후보로 사용해서
    H1 최대 threshold를 더 정확하게 찾는다.
    """
    scores = np.asarray(scores, dtype=float)

    thresholds = np.unique(
        np.r_[
            np.unique(scores),
            0.0,
            scores.min() - 1e-12,
            scores.max() + 1e-12
        ]
    )

    return thresholds


def is_better(candidate, current_best):
    """
    선택 기준:
    1순위 validation H1 최대
    2순위 validation G-Mean 최대
    3순위 validation F1 최대
    4순위 validation Recall 최대
    """
    if current_best is None:
        return True

    cand_key = (
        candidate["h1"],
        candidate["g_mean"],
        candidate["f1"],
        candidate["recall"],
    )

    best_key = (
        current_best["h1"],
        current_best["g_mean"],
        current_best["f1"],
        current_best["recall"],
    )

    return cand_key > best_key

### 지표

In [10]:
# =========================
# ParameterGrid + validation H1 기준 threshold search
# =========================
search_history = []
best_result = None

all_params = list(ParameterGrid(param_grid))

print(f"Total parameter combinations: {len(all_params)}")

for i, params in enumerate(all_params, start=1):

    model = Pipeline(
        steps=[("svm", SVC())]
    )
    model.set_params(**params)
    model.fit(X_train, y_train)

    valid_score_tmp = model.decision_function(X_valid)
    threshold_candidates = make_threshold_candidates(valid_score_tmp)

    best_for_params = None

    for threshold in threshold_candidates:
        valid_pred_tmp = (valid_score_tmp >= threshold).astype(int)
        metrics = compute_metrics(y_valid, valid_pred_tmp)

        result = {
            "params": params,
            "threshold": float(threshold),
            **{k: v for k, v in metrics.items() if k != "cm"},
        }

        if is_better(result, best_for_params):
            best_for_params = result

    search_history.append(best_for_params)

    if is_better(best_for_params, best_result):
        best_result = best_for_params

    print(
        f"[{i:>3}/{len(all_params)}] "
        f"H1={best_for_params['h1']:.4f}, "
        f"G-Mean={best_for_params['g_mean']:.4f}, "
        f"F1={best_for_params['f1']:.4f}, "
        f"Recall={best_for_params['recall']:.4f}, "
        f"Precision={best_for_params['precision']:.4f}, "
        f"Threshold={best_for_params['threshold']:.6f}, "
        f"Params={params}"
    )


# =========================
# Best model 재학습
# =========================
best_params = best_result["params"]
best_threshold = best_result["threshold"]
best_valid_h1 = best_result["h1"]

best_svm = Pipeline(
    steps=[("svm", SVC())]
)
best_svm.set_params(**best_params)
best_svm.fit(X_train, y_train)

# Validation prediction using selected threshold
valid_score = best_svm.decision_function(X_valid)
valid_pred = (valid_score >= best_threshold).astype(int)

valid_metrics = compute_metrics(y_valid, valid_pred)
cm_valid = valid_metrics["cm"]

valid_acc = valid_metrics["accuracy"]
valid_f1 = valid_metrics["f1"]
valid_recall = valid_metrics["recall"]
valid_precision = valid_metrics["precision"]
valid_g_mean = valid_metrics["g_mean"]
valid_h1 = valid_metrics["h1"]

print("\n" + "=" * 70)
print("Best SVM Model selected by validation H1 + decision_function threshold")
print("=" * 70)
print("Best Params    :", best_params)
print("Best Threshold :", best_threshold)
print("Best Valid H1  :", best_valid_h1)

print("\n[Validation] Metrics")
print(f"Accuracy  : {valid_acc:.4f}")
print(f"F1-Score  : {valid_f1:.4f}")
print(f"Recall    : {valid_recall:.4f}")
print(f"Precision : {valid_precision:.4f}")
print(f"G-Mean    : {valid_g_mean:.4f}")
print(f"H1-Score  : {valid_h1:.4f}")
print(f"\n[Validation] Confusion Matrix:\n{cm_valid}")

print("\nClassification Report:")
print(classification_report(y_valid, valid_pred, digits=4, zero_division=0))

# Search history 확인용
search_df = pd.DataFrame(search_history)
search_df = search_df.sort_values(
    by=["h1", "g_mean", "f1", "recall"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

search_df.head()

Total parameter combinations: 45
[  1/45] H1=0.4604, G-Mean=0.6453, F1=0.3578, Recall=0.5220, Precision=0.2722, Threshold=-0.990889, Params={'svm__C': 0.01, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  2/45] H1=0.4591, G-Mean=0.6447, F1=0.3565, Recall=0.5220, Precision=0.2707, Threshold=-0.910358, Params={'svm__C': 0.1, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  3/45] H1=0.4598, G-Mean=0.6362, F1=0.3600, Recall=0.4945, Precision=0.2830, Threshold=-0.635776, Params={'svm__C': 1, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  4/45] H1=0.4587, G-Mean=0.6713, F1=0.3483, Recall=0.6374, Precision=0.2397, Threshold=-0.835637, Params={'svm__C': 10, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  5/45] H1=0.4587, G-Mean=0.6713, F1=0.3483, Recall=0.6374, Precision=0.2397, Threshold=-0.835190, Params={'svm__C': 50, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  6/45] H1=0.4621, G-Mean=0.6445, F1=0.3602, Recall=0.5165, Precision=0.2765, Threshold=-0.992

,params,threshold,accuracy,f1,recall,precision,sensitivity,specificity,g_mean,h1,tn,fp,fn,tp
0,"{'svm__C': 0.01, 'svm__class_weight': None, 's...",-0.999600,0.767038,0.371482,0.543956,0.282051,0.543956,0.799363,0.659408,0.475236,1004,252,83,99
1,"{'svm__C': 0.01, 'svm__class_weight': None, 's...",-0.899187,0.791377,0.375000,0.494505,0.302013,0.494505,0.834395,0.642349,0.473546,1048,208,92,90
2,"{'svm__C': 10, 'svm__class_weight': None, 'svm...",-0.927773,0.791377,0.375000,0.494505,0.302013,0.494505,0.834395,0.642349,0.473546,1048,208,92,90
3,"{'svm__C': 10, 'svm__class_weight': None, 'svm...",-0.704132,0.791377,0.375000,0.494505,0.302013,0.494505,0.834395,0.642349,0.473546,1048,208,92,90
4,"{'svm__C': 50, 'svm__class_weight': None, 'svm...",-0.656585,0.791377,0.375000,0.494505,0.302013,0.494505,0.834395,0.642349,0.473546,1048,208,92,90


### Test 성능 결과 

In [11]:
# =========================
# Test evaluation using selected SVM model and selected threshold
# =========================
test_score = best_svm.decision_function(X_test)
test_pred = (test_score >= best_threshold).astype(int)

test_metrics = compute_metrics(y_test, test_pred)
cm_test = test_metrics["cm"]

test_acc = test_metrics["accuracy"]
test_f1 = test_metrics["f1"]
test_recall = test_metrics["recall"]
test_precision = test_metrics["precision"]
test_g_mean = test_metrics["g_mean"]
test_h1 = test_metrics["h1"]

print("\n" + "=" * 70)
print("[Test] Metrics")
print("=" * 70)
print("Selected Threshold:", best_threshold)
print(f"Accuracy  : {test_acc:.4f}")
print(f"F1-Score  : {test_f1:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"G-Mean    : {test_g_mean:.4f}")
print(f"H1-Score  : {test_h1:.4f}")
print(f"\n[Test] Confusion Matrix:\n{cm_test}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4, zero_division=0))


# =========================
# Save predictions
# =========================
result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

valid_pred_df = valid_df.copy()
valid_pred_df["SVM_Score"] = valid_score
valid_pred_df["SVM_Threshold"] = best_threshold
valid_pred_df["SVM_Pred"] = valid_pred

test_pred_df = test_df.copy()
test_pred_df["SVM_Score"] = test_score
test_pred_df["SVM_Threshold"] = best_threshold
test_pred_df["SVM_Pred"] = test_pred

valid_save_path = result_dir / "02. SVM_valid_pred.csv"
test_save_path = result_dir / "02. SVM_test_pred.csv"

valid_pred_df.to_csv(valid_save_path, index=False)
test_pred_df.to_csv(test_save_path, index=False)

print(f"\n✓ 저장 완료: {valid_save_path}")
print(f"✓ 저장 완료: {test_save_path}")


[Test] Metrics
Selected Threshold: -0.9996000042875016
Accuracy  : 0.7482
F1-Score  : 0.3213
Recall    : 0.4949
Precision : 0.2379
G-Mean    : 0.6225
H1-Score  : 0.4238

[Test] Confusion Matrix:
[[566 157]
 [ 50  49]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9188    0.7828    0.8454       723
           1     0.2379    0.4949    0.3213        99

    accuracy                         0.7482       822
   macro avg     0.5783    0.6389    0.5834       822
weighted avg     0.8368    0.7482    0.7823       822


✓ 저장 완료: ..\..\results\results_ML\02. SVM_valid_pred.csv
✓ 저장 완료: ..\..\results\results_ML\02. SVM_test_pred.csv


In [12]:
print("Best validation H1:", best_valid_h1)
print("Selected threshold:", best_threshold)
print("Test H1:", test_h1)

print("\nBest Params:")
print(best_params)

print("\nValid Confusion Matrix:")
print(confusion_matrix(y_valid, valid_pred, labels=[0, 1]))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))

Best validation H1: 0.47523639149344743
Selected threshold: -0.9996000042875016
Test H1: 0.4238415974636114

Best Params:
{'svm__C': 0.01, 'svm__class_weight': None, 'svm__gamma': 'auto', 'svm__kernel': 'poly'}

Valid Confusion Matrix:
[[1004  252]
 [  83   99]]

Test Confusion Matrix:
[[566 157]
 [ 50  49]]
